<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m2_01_train_test_error.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M2.01 — How strong will the concrete be?

In the lab we saw that a model can be perfect on the data it was trained
on and much worse on new data, and we used `ShuffleSplit`
cross-validation to measure the error we can really expect. In this
workbook you do the same **on laboratory tests of concrete**, and you will
see how far a single split can mislead you, when scaling matters and when
it does not, and a trap hidden in how the samples were made.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "higher_error": "9f86d081884c7d65",
    "honest_estimate": "053d4d953dfbf779",
    "ignored_feature": "625c722ac6b9f789",
    "more_stable": "a67edb5ffcb2be79",
    "n_features": "2c624232cdd22177",
    "n_test": "5cf4e26bd3d87da5",
    "target_column": "e6b88623f7209b6c"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data and target

Each row is one concrete sample made in a laboratory. The first seven
columns are the **recipe**: how many kilograms of each ingredient went
into one cubic metre of concrete. `age` is the number of days the sample
was left to harden. `strength` is the pressure (in megapascals, MPa) the
sample withstood before it broke.

A building company wants to know the strength **before** pouring the
concrete, so `strength` is what we predict.

In [ ]:
import pandas as pd

concrete = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/concrete_strength.csv"
)
concrete.head()

✏️ Fill in the name of the target column (a string) and the number of
features that remain once the target is set aside.

In [ ]:
target_column = ____
n_features = ____
check("target_column", target_column)
check("n_features", n_features)

data = concrete.drop(columns=target_column)
target = concrete[target_column]
print(f"{len(data)} samples, strength from {target.min()} to {target.max()} MPa, mean {target.mean():.1f}")

This is a **regression** problem: the target is a number, not a class. We
measure the quality of a model with the **mean absolute error** (MAE): on
average, by how many MPa is the prediction off? Lower is better.

## 2. A model that knows the answers

A decision tree with no limit on its size keeps splitting the data until
it can tell the training samples apart.

🔮 **Predict first.** We train such a tree on **all** the samples and then
ask it to predict the strength of **the same** samples. Will its error be
**"close to 0 MPa"**, **"about 5 MPa"** or **"about 15 MPa"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["close to 0 MPa", "about 5 MPa", "about 15 MPa"])

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state=0)
tree.fit(data, target)
error = mean_absolute_error(target, tree.predict(data))
print(f"error on the samples used for training: {error:.2f} MPa")

The tree has **memorised** the laboratory's notebook. That tells us
nothing about the next batch of concrete.

## 3. Train and test sets

✏️ Keep **a quarter** of the samples for testing: set `test_size` to that
fraction.

In [ ]:
from sklearn.model_selection import train_test_split

test_size = ____
data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=test_size, random_state=42
)
tree.fit(data_train, target_train)
train_error = mean_absolute_error(target_train, tree.predict(data_train))
test_error = mean_absolute_error(target_test, tree.predict(data_test))
print(f"training error: {train_error:.2f} MPa")
print(f"testing error:  {test_error:.2f} MPa")

✏️ On which set is the error **higher**? Write `"train"` or `"test"`.

In [ ]:
higher_error = ____
check("higher_error", higher_error)

🧠 **Think.** The company asks: "How wrong will your model be on the
concrete we pour next month?" Which of the two numbers do you give, and
why is the other one useless for this question?

## 4. Many splits instead of one

One test set is one draw. `ShuffleSplit` repeats the procedure: each time
it shuffles the samples, puts a share aside for testing, trains on the
rest.

✏️ Ask for **30** repetitions.

In [ ]:
from sklearn.model_selection import ShuffleSplit, cross_validate

n_splits = ____
cv = ShuffleSplit(n_splits=n_splits, test_size=0.2, random_state=0)
results = cross_validate(
    DecisionTreeRegressor(random_state=0),
    data,
    target,
    cv=cv,
    scoring="neg_mean_absolute_error",
    return_train_score=True,
)
train_errors = -results["train_score"]
test_errors = -results["test_score"]
print(f"training error: {train_errors.mean():.2f} ± {train_errors.std():.2f} MPa")
print(f"testing error:  {test_errors.mean():.2f} ± {test_errors.std():.2f} MPa")

scikit-learn scores follow the rule "higher is better", so it returns the
**negative** error; the minus signs above turn it back into an error.

✏️ There are 1030 samples and `test_size=0.2`. How many samples are in
**each** test set? Work it out, do not count them.

In [ ]:
n_test = ____
check("n_test", n_test, hint="Which share of the 1030 samples is put aside each time?")

The 30 testing errors are not all the same:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(test_errors, bins=10, edgecolor="black")
ax.set_xlabel("testing error (MPa)")
ax.set_ylabel("number of splits")
_ = ax.set_title("Testing error of the same model on 30 different splits")

Is an error of about 4 MPa good? A baseline helps, as in module 1. For
regression, `DummyRegressor` always predicts the mean strength of the
training set.

In [ ]:
from sklearn.dummy import DummyRegressor

dummy_errors = -cross_validate(
    DummyRegressor(), data, target, cv=cv, scoring="neg_mean_absolute_error"
)["test_score"]
print(f"dummy testing error: {dummy_errors.mean():.2f} ± {dummy_errors.std():.2f} MPa")
print(f"tree testing error:  {test_errors.mean():.2f} ± {test_errors.std():.2f} MPa")

### ➕ Going further: how much can one split mislead you?

Suppose 20 students each evaluate **the same tree on the same data**, and
the only thing that differs is the random seed of their split.

- Group A uses a **single** train/test split each.
- Group B uses **30 splits** each and reports the mean.

🔮 **Predict first.** In which group will the 20 reported errors disagree
the most? **"single split"**, **"30 splits"** or **"no difference"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["single split", "30 splits", "no difference"])

In [ ]:
single_split, thirty_splits = [], []
for seed in range(20):
    one = ShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    many = ShuffleSplit(n_splits=30, test_size=0.2, random_state=seed)
    for cv_seed, reported in [(one, single_split), (many, thirty_splits)]:
        scores = cross_validate(
            DecisionTreeRegressor(random_state=0),
            data,
            target,
            cv=cv_seed,
            scoring="neg_mean_absolute_error",
        )["test_score"]
        reported.append(-scores.mean())

reports = pd.DataFrame({"single split": single_split, "30 splits": thirty_splits})
reports.agg(["min", "max"]).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2.5))
ax.plot(single_split, [1] * 20, "o", alpha=0.6)
ax.plot(thirty_splits, [0] * 20, "o", alpha=0.6)
ax.set_yticks([0, 1], ["30 splits", "single split"])
ax.set_ylim(-0.5, 1.5)
_ = ax.set_xlabel("error reported by each of the 20 students (MPa)")

✏️ Which procedure gives almost the same answer whatever the seed? Write
`"single split"` or `"30 splits"`.

In [ ]:
more_stable = ____
check("more_stable", more_stable)

🧠 **Think.** Two students compare two different models, each with a
single split, and find 3.9 MPa against 4.4 MPa. From what you see above,
can they conclude that the first model is better?

## 5. ➕ Going further: does scaling matter here?

In workbook M1.02, a nearest-neighbours model broke down when age was
stored in days, because one feature decided the distances alone. Let us
look at this with our own eyes.

Builders care most about the strength after **28 days**, so we keep only
those samples, and two features:

- `slag`: kilograms of blast-furnace slag, between 0 and about 360;
- `water_cement_ratio`: kilograms of water divided by kilograms of cement,
  between about 0.3 and 1.9. Engineers have known for a century that more
  water per unit of cement gives weaker concrete.

In [ ]:
day28 = concrete[concrete["age"] == 28].copy()
day28["water_cement_ratio"] = day28["water"] / day28["cement"]
two_features = ["slag", "water_cement_ratio"]
data_28, target_28 = day28[two_features], day28["strength"]
data_28.describe().loc[["min", "max", "std"]].round(2)

A 10-nearest-neighbours model is trained on these two columns **without
scaling**. Think before looking at the result: which feature will it
practically ignore when it looks for neighbours?

✏️ Write the name of that column, exactly as it appears in the table
above.

In [ ]:
ignored_feature = ____
check("ignored_feature", ignored_feature, hint="Which feature has differences that are tiny next to the other one?")

The two maps below show what each model predicts for every combination of
the two features (yellow = strong, purple = weak). The dots are the real
samples.

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

models = {
    "without scaling": KNeighborsRegressor(n_neighbors=10),
    "with scaling": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10)),
}
slag_axis, ratio_axis = np.meshgrid(np.linspace(0, 360, 200), np.linspace(0.25, 1.9, 200))
grid = pd.DataFrame({"slag": slag_axis.ravel(), "water_cement_ratio": ratio_axis.ravel()})

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (name, model) in zip(axes, models.items()):
    model.fit(data_28, target_28)
    predicted = model.predict(grid).reshape(slag_axis.shape)
    image = ax.pcolormesh(slag_axis, ratio_axis, predicted, vmin=10, vmax=75)
    ax.scatter(
        data_28["slag"], data_28["water_cement_ratio"], c=target_28,
        vmin=10, vmax=75, edgecolor="black", linewidth=0.3, s=15,
    )
    ax.set_title(f"10 nearest neighbours {name}")
    ax.set_xlabel("slag (kg)")
axes[0].set_ylabel("water / cement ratio")
_ = fig.colorbar(image, ax=axes, label="strength (MPa)")

Without scaling the map is made of **vertical stripes**: the prediction
changes when you move left or right (slag) and hardly at all when you
move up or down (water/cement ratio). The model cannot see what the dots
show clearly: strong concrete is at the bottom.

Does that cost accuracy? We reuse 30 shuffled splits:

In [ ]:
cv_28 = ShuffleSplit(n_splits=30, test_size=0.2, random_state=0)
errors_28 = {}
for name, model in models.items():
    scores = cross_validate(model, data_28, target_28, cv=cv_28, scoring="neg_mean_absolute_error")
    errors_28[name] = -scores["test_score"].mean()
    print(f"{name:16s} {errors_28[name]:.2f} MPa")

The picture comes from scikit-learn's own example
[Importance of feature scaling](https://scikit-learn.org/1.8/auto_examples/preprocessing/plot_scaling_importance.html),
which does the same with wine data and a classifier.

### Is scaling always a win?

🔮 **Predict first.** We go back to **all** the samples and use `age`
(1 to 365 days) with `water_cement_ratio`. Again one feature has much
bigger numbers than the other. Will scaling make the error **"much
lower"**, **"about the same"** or **"much higher"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["much lower", "about the same", "much higher"])

In [ ]:
data_age = pd.DataFrame(
    {"age": concrete["age"], "water_cement_ratio": concrete["water"] / concrete["cement"]}
)
errors_age = {}
for name, model in models.items():
    scores = cross_validate(model, data_age, target, cv=cv, scoring="neg_mean_absolute_error")
    errors_age[name] = -scores["test_score"].mean()
    print(f"{name:16s} {errors_age[name]:.2f} MPa")

🧠 **Think.** Scaling changed what the model looked at in both
experiments. In the first it helped a lot, in the second it did not. What
is different about the feature with the big numbers in the two cases?
(Hint: how much does `slag` tell you about strength, and how much does
`age`?)

## 6. ➕ Going further: the same recipe on both sides

The laboratory did not make 1030 different concretes. It mixed a recipe,
poured several samples, and crushed them at different ages. Let us count
the distinct recipes (rows that share the same seven ingredients):

In [ ]:
ingredients = [column for column in data.columns if column != "age"]
recipe = concrete.groupby(ingredients).ngroup()
print(f"{len(concrete)} samples, {recipe.nunique()} distinct recipes")
recipe.value_counts().head()

With an ordinary shuffle, samples of one recipe land on **both** sides of
the split: the tree is tested on a recipe it has already seen, at another
age. `GroupShuffleSplit` keeps all the samples of a recipe together, so
every test recipe is new to the model.

🔮 **Predict first.** Compared with the ordinary shuffle, will the
testing error with whole recipes held out be **"lower"**, **"about the
same"** or **"higher"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["lower", "about the same", "higher"])

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

group_cv = GroupShuffleSplit(n_splits=30, test_size=0.2, random_state=0)
group_errors = -cross_validate(
    DecisionTreeRegressor(random_state=0),
    data,
    target,
    groups=recipe,
    cv=group_cv,
    scoring="neg_mean_absolute_error",
)["test_score"]
print(f"ordinary shuffle:       {test_errors.mean():.2f} ± {test_errors.std():.2f} MPa")
print(f"whole recipes held out: {group_errors.mean():.2f} ± {group_errors.std():.2f} MPa")

✏️ A client sends a recipe the laboratory has **never mixed** and asks how
far off the predicted strength may be. Which of the two estimates answers
that question? Write `"ordinary"` or `"grouped"`.

In [ ]:
honest_estimate = ____
check("honest_estimate", honest_estimate)

🧠 **Think.**

1. Neither number is "the wrong one". Describe a use of the model for
   which the **ordinary** estimate is the right one.
2. The data also contains 25 rows that are exact copies of another row.
   What happens when a copy is in the training set and its twin in the
   test set?
3. Before you split a dataset, what should you ask the people who
   collected it?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - why the training error of the tree says nothing about new concrete;
   - single split against 30 splits: what the 20 students would report;
   - the two maps: what the model without scaling cannot see, and why
     scaling did not help in the second experiment;
   - why holding out whole recipes gives a higher error, and which
     estimate you would give to which client.